In [ ]:
import copy
import os

import numpy as np
from barrier3d import Barrier3d
from matplotlib import pyplot as plt

from cascade.outwasher import Outwasher

# Outwash every 10 years

In [ ]:
b3d = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
years = np.load(
    "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years10.npy"
)
for t in range(1, 101):
    b3d.update()
    b3d.update_dune_domain()
    if b3d._time_index - 1 in years:
        outwash = Outwasher(
            datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
            outwash_years="outwash_years10.npy",
            outwash_bay_levels="outwash_baylevels10.npy",
            time_step_count=b3d._TMAX,
            berm_elev=b3d._BermEl,
            barrier_length=b3d._BarrierLength,
            sea_level=b3d._SL,
            bay_depth=-b3d._BayDepth,
            interior_domain=b3d.InteriorDomain,
            dune_domain=b3d.DuneDomain[b3d._time_index - 1],
            block_size=5,
            substep=60,
            sediment_flux_coefficient_Cx=10,
            sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
            max_slope=-0.25,
            shoreface_on=True,
        )
        outwash.update(b3d)
    print("B3D time step: ", b3d._time_index)

In [ ]:
outwash._Qs_shoreface[b3d._time_index - 1]  # m^3

In [ ]:
path = "D:/NC State/Outwasher/Output/newest_flow_routing/"
runID = "100years_outwash10_substep20"
newpath = path + runID + "/"
if not os.path.exists(newpath):
    os.makedirs(newpath)
os.chdir(newpath)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

In [ ]:
initial_domain = outwash._initial_full_domain
final_domain = outwash._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
# fig1.savefig(newpath + "0_domain", facecolor='w')

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)
# fig2.savefig(newpath + "final_domain", facecolor='w')

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-2,
    vmax=2,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)
# fig3.savefig(newpath + "elev_change_domain", facecolor='w')

In [ ]:
### HYDROGRAPH------------------------------------------------------------------------------------------------------------------
bay_levels = outwash._initial_bay_levels[0]
duration = len(bay_levels)
x = range(0, duration)

fig5 = plt.figure()
ax5 = fig5.add_subplot(111)
ax5.plot(x, bay_levels * 10, label="hydrograph")
ax5.set_xlabel("storm duration (hours)")
ax5.set_ylabel("m MHW")
ax5.set_title("bay elevation over the course of each storm")
# ax5.legend(loc="lower right")
plt.show()
# fig5.savefig(newpath + "hydrograph", facecolor='w')
plt.close()

In [ ]:
m_xsTS = np.subtract(b3d.x_s_TS, b3d.x_s_TS[0])
m_xsTS = np.multiply(m_xsTS, 10)

# B3D Only

In [ ]:
b3d2 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
b3d2_interior = np.zeros(101)
for t in range(1, 101):
    b3d2.update()
    b3d2.update_dune_domain()
    print("B3D time step: ", b3d2._time_index)

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

m_xsTS2 = np.subtract(b3d2.x_s_TS, b3d2.x_s_TS[0])
m_xsTS2 = np.multiply(m_xsTS2, 10)
plt.plot(m_xsTS2, label="B3D only")
plt.plot(m_xsTS, label="Washout to shoreface")
# plt.vlines(years3, ymin=min(m_xsTS), ymax=80, colors='k', linestyles='dotted', linewidth=1)
plt.xlabel("years")
plt.ylabel("shoreline position (m)")
plt.ylim([0, 80])
plt.legend()
# plt.title("Outwash to Shoreface")
# plt.savefig(r"D:\NC State\Outwasher\Output\newest_flow_routing/b3d_outwash10_shorelinechanges", facecolor='w')

# Outwash out of system

In [ ]:
b3d3 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
years3 = np.load(
    "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years10.npy"
)

for t in range(1, 101):
    b3d3.update()
    b3d3.update_dune_domain()
    if b3d3._time_index - 1 in years3:
        outwash3 = Outwasher(
            datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
            outwash_years="outwash_years10.npy",
            outwash_bay_levels="outwash_baylevels10.npy",
            time_step_count=b3d3._TMAX,
            berm_elev=b3d3._BermEl,
            barrier_length=b3d3._BarrierLength,
            sea_level=b3d3._SL,
            bay_depth=-b3d3._BayDepth,
            interior_domain=b3d3.InteriorDomain,
            dune_domain=b3d3.DuneDomain[b3d3._time_index - 1],
            block_size=5,
            substep=60,
            sediment_flux_coefficient_Cx=10,
            sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
            max_slope=-0.25,
            shoreface_on=False,
        )
        outwash3.update(b3d3)

    print("B3D time step: ", b3d3._time_index)

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

initial_domain = outwash3._initial_full_domain
final_domain = outwash3._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
# fig1, (ax1, ax3) = plt.subplots(1, 2, sharey=True)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
# fig1.savefig(newpath + "0_domain", facecolor='w')

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)
# fig2.savefig(newpath + "final_domain", facecolor='w')

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-2,
    vmax=2,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)
# fig3.savefig(newpath + "elev_change_domain", facecolor='w')

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)

m_xsTS3 = np.subtract(b3d3.x_s_TS, b3d3.x_s_TS[0])
m_xsTS3 = np.multiply(m_xsTS3, 10)
# plt.vlines(years3-1, ymin=min(m_xsTS3), ymax=max(m_xsTS3), colors='r', linestyles='dotted')
plt.vlines(
    years3 - 1, ymin=min(m_xsTS3), ymax=80, colors="k", linestyles="dotted", linewidth=1
)

plt.plot(m_xsTS2, label="B3D only")
plt.plot(m_xsTS, label="Washout to shoreface")
plt.plot(m_xsTS3, label="Washout lost")

plt.ylim([0, 80])
plt.xlabel("years")
plt.ylabel("shoreline position (m)")
plt.legend()
# plt.savefig(r"D:\NC State\Outwasher\Output\newest_flow_routing\shoreface_nourishment_comparison_substep20.png", facecolor='w')

In [ ]:
OWTS = b3d.QowTS  # m3/m
OWTS2 = b3d2.QowTS  # m3/m
OWTS3 = b3d3.QowTS  # m3/m

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams.update({"font.size": 15})

plt.plot(OWTS2, label="B3D only")
plt.plot(OWTS, linestyle="dashed", label="Washout to shoreface")
# plt.plot(OWTS3, label="Outwash every 20 years")
# plt.vlines(years3-1, ymin=min(m_xsTS3), ymax=max(m_xsTS3), colors='r', linestyles='dotted')
# plt.vlines(years-1, ymin=min(m_xsTS), ymax=max(m_xsTS), colors='k', linestyles='dotted')
plt.xlabel("years")
plt.ylabel("overwash flux ($m^3/m$)")
plt.legend()
# plt.title("Outwash to Shoreface")
# plt.savefig(r"D:\NC State\Outwasher\Output\newest_flow_routing\overwash_comparison_substep20.png", facecolor='w')